In [3]:
import io
import zipfile
import requests
import frontmatter
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from tqdm.auto import tqdm

OPENAI_APIKEY = os.getenv('OPENAI_API_KEY')

In [4]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/master'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [5]:
# tensor_faq = read_repo_data('emmanuellar', 'Community-Management-Resources')
# microsoft_docs = read_repo_data('microsoft', 'AI-For-Beginners')
ml_docs = read_repo_data('DataTalksClub', 'machine-learning-zoomcamp')

# print(f"Community documents: {len(tensor_faq)}")
# print(f"Microsoft documents: {len(microsoft_docs)}")
print(f"ML Zoomcamp documents: {len(ml_docs)}")

ML Zoomcamp documents: 226


## Simple Chunking


In [6]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [7]:
ml_chunks = []

for doc in ml_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    ml_chunks.extend(chunks)

In [8]:
print(ml_chunks)

[{'start': 0, 'chunk': '## 1.1 Introduction to Machine Learning\n\n<a href="https://www.youtube.com/watch?v=Crm_5n4mvmg&list=PL3MmuxUbc_hIhxl5Ji8t4O6lPAOpHaCLR&index=2"><img src="images/thumbnail-1-01.jpg"></a>\n\n[Slides](https://www.slideshare.net/AlexeyGrigorev/ml-zoomcamp-11-introduction-to-machine-learning)\n\n\n## Notes\n\nThe concept of ML is depicted with an example of predicting the price of a car. The ML model\nlearns from data, represented as some **features** such as year, mileage, among others, and the **target** variable, in this\ncase, the car\'s price, by extracting patterns from the data.\n\nThen, the model is given new data (**without** the target) about cars and predicts their price (target). \n\nIn summary, ML is a process of **extracting patterns from data**, which is of two types:\n\n* features (information about the object) and \n* target (property to predict for unseen objects). \n\nTherefore, new feature values are presented to the model, and it makes **predict

## Section Based Chunking

In [9]:
import re
text = ml_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [10]:
def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections

In [11]:
sections = split_markdown_by_level(text, level=2)
# print(sections)

In [12]:
ml_chunks = []

for doc in ml_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        ml_chunks.append(section_doc)

## AI Based Chunking (Intelligent)

In [13]:
openai_client = OpenAI()


def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )

    return response.output_text

In [14]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [15]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [16]:
ml_chunks = []

for doc in tqdm(ml_docs):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        ml_chunks.append(section_doc)


  0%|          | 0/226 [00:00<?, ?it/s]